# Training and Generalization

고정 split에서 capacity를 비교하고 learning rate의 안정 범위와 best checkpoint를 검사합니다.

핵심 셀은 Python 3 + NumPy만 사용합니다. 데이터 다운로드와 API 키가 필요하지 않습니다.

## 고정된 noisy 입력과 split

In [ ]:
import numpy as np
rng=np.random.default_rng(42)
x=np.linspace(-1,1,28); y=np.sin(np.pi*x)+rng.normal(0,.15,len(x))
valid=np.arange(0,len(x),4); train=np.setdiff1d(np.arange(len(x)),valid)
assert not set(train)&set(valid)

## capacity 비교

In [ ]:
records=[]
for degree in [1,3,9,16]:
    coef=np.polynomial.polynomial.polyfit(x[train],y[train],degree)
    tr=np.mean((np.polynomial.polynomial.polyval(x[train],coef)-y[train])**2)
    va=np.mean((np.polynomial.polynomial.polyval(x[valid],coef)-y[valid])**2)
    records.append((degree,tr,va))
print(np.array(records))
selected=min(records,key=lambda a:a[2])[0]
print("validation-selected degree",selected)

## scalar gradient 안정성

In [ ]:
def run(rate,steps=12):
    w=4.; hist=[]; best=(float("inf"),None)
    for step in range(steps):
        loss=(w-1)**2
        hist.append(loss)
        if loss<best[0]: best=(loss,float(w))
        w-=rate*2*(w-1)
    return np.array(hist),best
for rate in [.2,.8,1.,1.1]:
    hist,best=run(rate)
    print(rate,"first/last",hist[0],hist[-1],"best",best)

## 과제

- η=0.2와0.8의 loss trace가 같은지 확인하고 error multiplier로 설명하세요.
- best checkpoint를 current weight와 독립된 값으로 보존하고 large-rate run에서 마지막 state와 비교하세요.

## 해답과 검증

In [ ]:
h1,b1=run(.2); h2,b2=run(.8)
assert np.allclose(h1,h2)
hbad,bbad=run(1.1)
assert hbad[-1]>hbad[0] and bbad[0]==hbad[0]
print("stable/oscillating traces verified; independent best state",bbad)